# 1. Reshaping Training and Test Data

In [ ]:
import numpy as np

X_train = np.load("/Users/mayawiegand/Documents/ECS 171/Project/music-genre-classification/processed_data/X_train.npy")
X_test = np.load("/Users/mayawiegand/Documents/ECS 171/Project/music-genre-classification/processed_data/X_test.npy")
y_train = np.load("/Users/mayawiegand/Documents/ECS 171/Project/music-genre-classification/processed_data/y_train.npy")
y_test = np.load("/Users/mayawiegand/Documents/ECS 171/Project/music-genre-classification/processed_data/y_test.npy")

print(X_train.shape) # spectrogram is samples, number of mel filters (n_mels), number of time frames
# need to flatten this as KNN can't take 3D data

X_train = X_train.reshape(X_train.shape[0], -1) # this transforms the data to number of samples, n_mels * n time frames
X_test = X_test.reshape(X_test.shape[0], -1) 

print(X_train.shape)


# 2. Label Encoding
- Needed since some scikit-learn functions require numeric-based class labels rather than strings/categorical
- Converts categorical class labels to integers so they can be processed correctly
    - Not the same as encoding features, as this doesn't influence model predictions

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train) # fitting on the training, and applying this transformation
y_test_enc  = le.transform(y_test) # applying transformation only (using the training fit)

# 3. Fitting Baseline KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=50) # initializing KNN model with k = 50

knn.fit(X_train, y_train_enc) # fitting model on training data

y_pred_enc = knn.predict(X_test) # forming predictions by applying fitted model

y_pred_labels = le.inverse_transform(y_pred_enc) # converting the encoded labels back to the genre names
print(y_pred_labels)

In [ ]:
# quick analysis measure

from sklearn.metrics import f1_score
f1 = f1_score(y_test, y_pred_labels, average='macro') # using labels so true and predicted values match in datatype
f1 # supper low, tuning hyperparameters should help with this

# 5. Hyperparameter Tuning Setup

In [ ]:
from sklearn.model_selection import GridSearchCV

# defining the grid of values we want to test
search_space = {
    "n_neighbors": list(range(1,81)),
    "weights": ["uniform", "distance"],
    "metric": ["manhattan", "euclidean"]
}

grid_search = GridSearchCV(estimator = knn,
                           param_grid = search_space,
                           scoring = "f1_macro",
                           cv = 5,
                           n_jobs=-1 # using all CPU cores to speed up grid search
)

grid_search.fit(X_train, y_train_enc)

In [ ]:
grid_search.best_params_

# 6. PCA
- Will also perform PCA, a dimension-reduction technique and compare model performance in both non-PCA predictions and PCA predictions

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components = 0.95) # defining PCA using the optimal number of components (the number that explain 95% of variance observed)

# transforming the training and testing data using PCA
X_train_pca = pca.fit_transform(X_train) 
X_test_pca = pca.transform(X_test)

n_components_kept = pca.n_components_ 
print(n_components_kept) # prints the number of components kept = explain 95% of variance

# 7. Hyperparameter Tuning with PCA

In [ ]:
# since PCA changes the dimension the data is in, need to re-perform hyperparameter tuning for dimension-reduced data

grid_search_pca = GridSearchCV(estimator = knn,
                           param_grid = search_space,
                           scoring = "f1_macro",
                           cv = 5,
                           n_jobs=-1
)

grid_search_pca.fit(X_train_pca, y_train_enc) # running grid search on PCA data

In [ ]:
grid_search_pca.best_params_

# Next steps
- Analyze the hyperparamter tuning - plot of k vs F1
- Fit non-PCA and PCA data with best model, compare performance metrics, generate model predictions and analyze
- May also want to add more performance metrics for hyperparameter tuning (accuracy, precision, etc)
- Can also switch to Bayesian SearchCV if Grid SearchCV is too computationally expensive
    - Doesn't search through every combination like grid search does, but updates the probability distributions of the combinations of hyperparameters to try to reach optimal solution
    - Though isn't guarenteed to converge to best combo
- Interpret results: plots by genre based on the percentage of correct predictions, research into literature as to why some genres perform better than others
    - F1 score for each of the genres
- Does PCA data perform better? Why is this/why does this make sense?